# MDM2 Fixed Pipeline — Train + Screen (Colab-ready)
Fresh notebook, no deprecated APIs. Fixes: scaffold split, single featurizer for train+screen, calibrated RF, vectorized screening, AD + T1/T2/T3 tiers (RF never hard-vetoes regression/novel hits), no PRED_CAP.

**Run top-to-bottom on Colab T4. Expects `ro5_properties_filtered.csv` (645 train) next to notebook or auto-falls back to repo copy. COCONUT library: uses your already-filtered/reduced file if present, else `screening_results.csv` SMILES, else downloads paper COCONUT.**

In [ ]:
# 0 — Colab setup (safe to re-run)
import os, sys
IN_COLAB = os.path.isdir('/content')
print('IN_COLAB =', IN_COLAB)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable,'-m','pip','install','--quiet','rdkit','scikit-learn>=1.3','pandas','numpy','matplotlib','joblib','gdown'], check=False)
import sklearn, numpy, pandas, rdkit
print('sklearn', sklearn.__version__, '| rdkit', rdkit.__version__)

In [ ]:
# 1 — Imports + config (single source of truth)
import os, joblib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit, RepeatedStratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix
RANDOM_SEED=57; np.random.seed(RANDOM_SEED)
MORGAN_RADIUS=3; MORGAN_NBITS=512   # <-- used for TRAIN and SCREEN (consistency fix)
AD_CUT=0.35; RF_T1_CUT=0.6; RF_T2_CUT=0.4; GNN_HIT_CUT=7.5  # T1=GNN>=cut & RF>T1 & AD; T2=novel/rescue
TRAIN_CANDIDATES=['ro5_properties_filtered.csv','Part_2/ro5_properties_filtered.csv','Part_15/ro5_properties_filtered.csv']
COCONUT_CANDIDATES=['coconut_reduced.csv','coconut_filtered.csv','ro5_coconut_filtered.csv','Part_8/screening_results.csv','screening_results.csv']
print('config OK:', MORGAN_RADIUS, MORGAN_NBITS)

In [ ]:
# 2 — Load TRAIN (645 ChEMBL, Ro5 already passed)
TRAIN_CANDIDATES2 = TRAIN_CANDIDATES + ['/content/ro5_properties_filtered.csv']
TRAIN_PATH = next((p for p in TRAIN_CANDIDATES2 if os.path.exists(p)), None)
if TRAIN_PATH is None:
    import urllib.request
    print('downloading train from GitHub...')
    TRAIN_PATH='/tmp/ro5_properties_filtered.csv'
    urllib.request.urlretrieve('https://raw.githubusercontent.com/Techbjd/new_method/main/Part_2/ro5_properties_filtered.csv', TRAIN_PATH)
print('TRAIN =', TRAIN_PATH)
df=pd.read_csv(TRAIN_PATH)
print(df.shape); print(df['bioactivity_class'].value_counts().to_dict())
print('pIC50 range:', float(df.pIC50.min()), float(df.pIC50.max()))
df['label']=(df['bioactivity_class']=='Active').astype(int)
display(df.head(3))


In [ ]:
# 3 — ONE featurizer for everything (modern API, no deprecated GetMorganFingerprintAsBitVect)
mpg = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_NBITS)
def smiles_to_fp(smiles):
    m=Chem.MolFromSmiles(str(smiles))
    if m is None: return None
    return np.array(mpg.GetFingerprint(m), dtype=np.uint8)
# sanity: 5 train SMILES
X0=np.stack([smiles_to_fp(s) for s in df.canonical_smiles.head(5)])
print('fp shape:', X0.shape, '| n_on avg:', X0.sum(axis=1))
assert X0.shape[1]==MORGAN_NBITS

In [ ]:
# 4 — Featurize full train (vectorized, invalid-SMILES safe)
assert 'df' in globals(), 'Run cell #2 first (df missing). Run top-to-bottom.'
assert 'smiles_to_fp' in globals(), 'Run cell #3 first (featurizer missing). Run top-to-bottom.'
print('columns:', list(df.columns))
SMILES_COL = 'canonical_smiles' if 'canonical_smiles' in df.columns else ('smiles' if 'smiles' in df.columns else None)
assert SMILES_COL is not None, f'No SMILES column! Found: {list(df.columns)}'
fps=[]; keep=[]; bad=0
for i,s in enumerate(df[SMILES_COL]):
    try: v=smiles_to_fp(s)
    except Exception: v=None
    if v is not None: fps.append(v); keep.append(i)
    else: bad+=1
assert len(fps)>0, f'All {len(df)} SMILES failed! First 3: {list(df[SMILES_COL].head(3))}'
X=np.stack(fps); y=df['label'].values[keep]
smiles_train=df[SMILES_COL].values[keep]
print('kept', X.shape, 'dropped', bad)


In [ ]:
# 5 — Scaffold split PRIMARY (random = optimistic appendix only)
def murcko(s):
    try:
        m=Chem.MolFromSmiles(str(s))
        return MurckoScaffold.MurckoScaffoldSmiles(m) if m else 'INVALID'
    except Exception: return 'INVALID'
scaffolds=np.array([murcko(s) for s in smiles_train])
print('unique scaffolds:', len(set(scaffolds)))
gss=GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
try:
    tr,te=next(gss.split(X,y,groups=scaffolds))
    print('scaffold split OK')
except Exception as e:
    print('scaffold split failed, fallback to stratified random:', e)
    tr,te=next(StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=RANDOM_SEED).split(X,y))
Xtr,Xte,ytr,yte=X[tr],X[te],y[tr],y[te]
print('train',Xtr.shape, ytr.mean().round(3), '| test',Xte.shape, yte.mean().round(3))


In [ ]:
# 6 — Train RF + calibrate (no PRED_CAP, no delta trick here)
base=RandomForestClassifier(n_estimators=300, max_depth=None, min_samples_leaf=2, class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)
grid=GridSearchCV(base, {'max_depth':[None,20],'min_samples_leaf':[1,2]}, cv=3, scoring='roc_auc', n_jobs=-1)
grid.fit(Xtr,ytr)
print('best:', grid.best_params_, round(grid.best_score_,4))
rf=grid.best_estimator_
cal=CalibratedClassifierCV(rf, method='sigmoid', cv=5)
cal.fit(Xtr,ytr)
for name,model in [('RF',rf),('RF-cal',cal)]:
    p=model.predict_proba(Xte)[:,1]
    print(f"{name}: AUC={roc_auc_score(yte,p):.3f} AP={average_precision_score(yte,p):.3f}")
print(classification_report(yte, rf.predict(Xte), target_names=['inact','act']))
print(confusion_matrix(yte, rf.predict(Xte)))
joblib.dump(rf,'rf_mdm2_512.pkl'); joblib.dump(cal,'rf_mdm2_512_cal.pkl')
print('saved pkl (512 feats)')

In [ ]:
# 7 — Honesty: repeated-CV + 3x y-shuffle (fast)
cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_SEED)
auc=cross_val_score(rf,X,y,cv=cv,scoring='roc_auc',n_jobs=-1)
print(f'7-fold-equiv repeated CV AUC: {auc.mean():.3f} +- {auc.std():.3f}')
for r in range(3):
    ys=np.random.RandomState(r).permutation(y)
    a=cross_val_score(rf,X,ys,cv=3,scoring='roc_auc',n_jobs=-1).mean()
    print(f'shuffle {r}: AUC={a:.3f} (expect ~0.5)')

In [ ]:
# 8 — Load COCONUT library (your already-reduced file first)
LIB=None
for p in COCONUT_CANDIDATES+['testing/fix_of_chemphore/screening_results.csv','/content/coconut_csv-03-2025.csv']:
    if os.path.exists(p): LIB=p; break
print('LIB =', LIB)
if LIB is None or os.path.isdir('/content' ) and False:
    pass
if LIB is None:
    import gdown
    gdown.download(id='1-DFc6lMf6maNAWPwZFA8uY6951SokoNY', output='/tmp/coconut_csv-03-2025.csv', quiet=False)
    LIB='/tmp/coconut_csv-03-2025.csv'
lib=pd.read_csv(LIB, usecols=lambda c: c.lower() in ('canonical_smiles','smiles','identifier','molecule_chembl_id'))
print(lib.shape, list(lib.columns))
# normalize col names
low={c:c.lower() for c in lib.columns}
lib=lib.rename(columns={c:('canonical_smiles' if 'smil' in low[c] else ('identifier' if 'ident' in low[c] or 'chembl' in low[c] else c)) for c in lib.columns})
if 'identifier' not in lib.columns: lib['identifier']=['LIB_%d'%i for i in range(len(lib))]
lib=lib.dropna(subset=['canonical_smiles']).drop_duplicates('canonical_smiles').reset_index(drop=True)
print('dedup:', lib.shape)
display(lib.head(3))

In [ ]:
# 9 — Screen (vectorized RF + AD Tanimoto to train actives, chunked)
N=len(lib)
BATCH=20000
probs=np.zeros(N); preds=np.zeros(N,dtype=int)
for s in range(0,N,BATCH):
    e=min(N,s+BATCH)
    F=np.stack([smiles_to_fp(x) if smiles_to_fp(x) is not None else np.zeros(MORGAN_NBITS,np.uint8) for x in lib.canonical_smiles[s:e]])
    probs[s:e]=cal.predict_proba(F)[:,1]; preds[s:e]=(probs[s:e]>=0.5).astype(int)
    print(f'{e}/{N}', flush=True)
# AD: max Tanimoto vs 200 sampled train actives (fast, honest)
from rdkit.DataStructs import BulkTanimotoSimilarity
act_mols=[Chem.MolFromSmiles(s) for s in smiles_train[y==1][:200]]
act_fps=[mpg.GetFingerprint(m) for m in act_mols if m]
maxsim=np.zeros(N)
for s in range(0,N,BATCH):
    e=min(N,s+BATCH)
    for i,sm in enumerate(lib.canonical_smiles[s:e]):
        m=Chem.MolFromSmiles(str(sm))
        maxsim[s+i]=max(BulkTanimotoSimilarity(mpg.GetFingerprint(m), act_fps)) if m else 0.0
lib['rf_prob']=probs; lib['rf_pred']=preds; lib['maxsim_train']=maxsim
lib['in_AD']=lib['maxsim_train']>=AD_CUT
print(lib.rf_prob.describe().round(4).to_dict(), '| in_AD frac:', round(float(lib.in_AD.mean()),4))

In [ ]:
# 10 — Tiers (RF never hard-vetoes novelty) + optional GNN merge
# if you have regression preds (pred_pic50), drop file 'coconut_regression.csv' [identifier,pred_pic50] next to notebook to merge
import glob
reg=None
for p in ['coconut_regression.csv','/content/coconut_regression.csv']:
    if os.path.exists(p): reg=pd.read_csv(p); break
if reg is not None:
    lib=lib.merge(reg[['identifier','pred_pic50']], on='identifier', how='left')
    print('merged regression:', reg.shape)
else:
    lib['pred_pic50']=np.nan; print('no regression file — tiers use RF+AD only')
def tier(r):
    g = r['pred_pic50'] if pd.notna(r['pred_pic50']) else np.nan
    if pd.notna(g) and g>=GNN_HIT_CUT and r['rf_prob']>=RF_T1_CUT and r['in_AD']: return 'T1'
    if r['rf_prob']>=RF_T1_CUT and r['in_AD']: return 'T1'
    if (pd.notna(g) and g>=GNN_HIT_CUT) or r['rf_prob']>=RF_T2_CUT: return 'T2'
    return 'T3'
lib['tier']=lib.apply(tier,axis=1)
print(lib.tier.value_counts().to_dict())
for t in ['T1','T2','T3']:
    sub=lib[lib.tier==t].nlargest(50,'rf_prob')
    sub.to_csv(f'hits_{t}.csv',index=False); print(t, sub.shape)
lib.to_csv('screening_full_fixed.csv',index=False); print('saved screening_full_fixed.csv')

In [ ]:
# 11 — Report: top-15 + plots
display(lib.sort_values('rf_prob',ascending=False).head(15)[['identifier','rf_prob','maxsim_train','tier']])
fig,ax=plt.subplots(1,2,figsize=(11,3.5))
ax[0].hist(lib.rf_prob, bins=60); ax[0].set_title('RF-cal prob (screen)'); ax[0].axvline(0.5,c='r',ls='--'); ax[0].axvline(0.6,c='g',ls='--')
ax[1].hist(lib.maxsim_train, bins=60); ax[1].set_title('AD maxsim to train'); ax[1].axvline(AD_CUT,c='r',ls='--')
plt.tight_layout(); plt.savefig('screening_QC.png',dpi=120); plt.show()
print('QC saved. Dock T1 first (4HG7), T2 = novelty appendix, T3 = OOD/reject.')

In [ ]:
# 12 — GNN setup (GCN + GIN from Part_18, same 645 train)
import sys, urllib.request, os
GH='https://raw.githubusercontent.com/Techbjd/new_method/main'
!pip install --quiet torch torch_geometric rdkit 2>&1 | tail -1
os.makedirs('/tmp/gnn', exist_ok=True)
for f in ['Part_18_GNN_Triage/common.py','Part_18_GNN_Triage/models/gcn_fixed.pth','Part_18_GNN_Triage/models/gin_fixed.pth']:
    print('get', f)
    urllib.request.urlretrieve(f"{GH}/{f}", '/tmp/gnn/'+f.split('/')[-1])
sys.path.insert(0, '/tmp/gnn')
import torch
from common import GCN, mol_to_graph
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())


In [ ]:
# 13 — GNN score dock list (RF + GNN consensus, no 695k rerun)
import pandas as pd, numpy as np, torch
from torch_geometric.loader import DataLoader
from common import mol_to_graph, GCN
INP = 'dock_T1_ready.csv' if os.path.exists('dock_T1_ready.csv') else 'hits_T1.csv'
print('GNN input:', INP)
t = pd.read_csv(INP)
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def load_gnn(w):
    m = GCN(num_node_features=78, hidden_dim=128, num_classes=2)
    m.load_state_dict(torch.load(w, map_location='cpu')); m.to(dev).eval()
    return m
gcn = load_gnn('/tmp/gnn/gcn_fixed.pth')
graphs = [g for g in (mol_to_graph(s) for s in t['canonical_smiles']) if g is not None]
print('graphs:', len(graphs), '/', len(t))
loader = DataLoader(graphs, batch_size=64, shuffle=False)
probs = []
import torch.nn.functional as F
with torch.no_grad():
    for b in loader:
        b = b.to(dev)
        probs += F.softmax(gcn(b.x, b.edge_index, b.batch), dim=1)[:,1].cpu().tolist()
t['gnn_prob'] = probs + [float('nan')]*(len(t)-len(probs))
t['consensus'] = (t['rf_prob'] + t['gnn_prob'])/2
t = t.sort_values('consensus', ascending=False)
t.to_csv('dock_T1_consensus.csv', index=False)
display(t.head(10)[['identifier','rf_prob','gnn_prob','consensus']])
print('saved dock_T1_consensus.csv — dock by consensus desc')
